In [1]:
import pandas as pd
from pathlib import Path

In [2]:
snp_df = pd.read_csv("../output/plotting_data/snp/combined_comparison_summary_long.tsv", sep="\t")

no_snp_df = pd.read_csv("../output/plotting_data/quality/dna-contained/combined_comparison_summary_long.tsv", sep="\t")



In [3]:
no_snp_df = no_snp_df[no_snp_df["k"] == 15]
no_snp_df

,k,offset,threshold,mode,gene,filtered_reads,mapping_reads,matched,not_mapped,not_filtered,summary_file
1600,15,15,75,and,ENSSSCG00000000393,13411,3909,3892,9519,17,k_15/offset_15/threshold_75/mode_and/compariso...
1601,15,15,75,and,ENSSSCG00000000482,0,0,0,0,0,k_15/offset_15/threshold_75/mode_and/compariso...
1602,15,15,75,and,ENSSSCG00000000540,19564,0,0,19564,0,k_15/offset_15/threshold_75/mode_and/compariso...
1603,15,15,75,and,ENSSSCG00000000610,66,3,3,63,0,k_15/offset_15/threshold_75/mode_and/compariso...
1604,15,15,75,and,ENSSSCG00000000665,8242,13,13,8229,0,k_15/offset_15/threshold_75/mode_and/compariso...
...,...,...,...,...,...,...,...,...,...,...,...
2395,15,15,120,or,ENSSSCG00000060338,18041,0,0,18041,0,k_15/offset_15/threshold_120/mode_or/compariso...
2396,15,15,120,or,ENSSSCG00000062444,11033,999,949,10084,50,k_15/offset_15/threshold_120/mode_or/compariso...
2397,15,15,120,or,ENSSSCG00000062464,9707,193,192,9515,1,k_15/offset_15/threshold_120/mode_or/compariso...
2398,15,15,120,or,ENSSSCG00000063089,0,0,0,0,0,k_15/offset_15/threshold_120/mode_or/compariso...


In [7]:
df = snp_df.merge(
    no_snp_df,
    on=["mode", "threshold", "gene", "offset", "k"],
    how="inner",
)
df = df.drop(columns=["summary_file_x", "summary_file_y"])
df["filtered_reads_dif"] = df["filtered_reads_x"] - df["filtered_reads_y"]
df["matched_dif"] = df["matched_x"] - df["matched_y"]
df["not_mapped_dif"] = df["not_mapped_x"] - df["not_mapped_y"]
df["not_filtered_dif"] = df["not_filtered_x"] - df["not_filtered_y"]
df = df[["mode", "threshold", "gene", "offset", "k", "filtered_reads_dif", "matched_dif", "not_mapped_dif", "not_filtered_dif"]]

In [8]:
df = df.sort_values(by=["filtered_reads_dif"], axis=0, ascending=False)
df

,mode,threshold,gene,offset,k,filtered_reads_dif,matched_dif,not_mapped_dif,not_filtered_dif
176,or,75,ENSSSCG00000039161,15,15,175726,0,175726,0
102,or,75,ENSSSCG00000000540,15,15,165364,0,165364,0
170,or,75,ENSSSCG00000036095,15,15,164009,0,164009,0
376,or,90,ENSSSCG00000039161,15,15,131120,0,131120,0
302,or,90,ENSSSCG00000000540,15,15,121435,0,121435,0
...,...,...,...,...,...,...,...,...,...
194,or,75,ENSSSCG00000060126,15,15,0,0,0,0
193,or,75,ENSSSCG00000058896,15,15,0,0,0,0
192,or,75,ENSSSCG00000058088,15,15,0,0,0,0
350,or,90,ENSSSCG00000016949,15,15,0,0,0,0


In [9]:
df.mean(numeric_only=True)

threshold               97.50000
offset                  15.00000
k                       15.00000
filtered_reads_dif    3237.62375
matched_dif              0.03750
not_mapped_dif        3237.58625
not_filtered_dif        -0.03750
dtype: float64

In [10]:
df.median(numeric_only=True)

threshold              97.5
offset                 15.0
k                      15.0
filtered_reads_dif    154.5
matched_dif             0.0
not_mapped_dif        154.5
not_filtered_dif        0.0
dtype: float64

In [11]:
df.max(numeric_only=True)

threshold                120
offset                    15
k                         15
filtered_reads_dif    175726
matched_dif                5
not_mapped_dif        175726
not_filtered_dif           0
dtype: int64

In [12]:
kmer_dir = Path("../output/kmers_pig")
gene_to_kmer_count = {
    file.stem: max(sum(1 for _ in file.open()) - 1, 0)
    for file in sorted(kmer_dir.glob("*.tsv"))
}

In [20]:
kmer_count_df = pd.DataFrame(gene_to_kmer_count.items(), columns=["gene", "kmer_count"])
df_count = df.merge(kmer_count_df, on="gene", how="inner")
df_count[["filtered_reads_dif", "kmer_count"]].corr(method="spearman")

,filtered_reads_dif,kmer_count
filtered_reads_dif,1.000000,0.668962
kmer_count,0.668962,1.000000


In [14]:
snp_run = pd.read_csv("../output/plotting_data/snp/time_summary.tsv", sep="\t")
dna_run = pd.read_csv("../time_summary.tsv", sep="\t")
read_counts = pd.read_csv("../output/read_counts.tsv", sep="\t")

In [15]:
dna_run_fil = dna_run[dna_run["k"] == 15]
dna_run_fil = dna_run_fil[dna_run_fil["offset"] == 15]
dna_run_fil = dna_run_fil[dna_run_fil["threshold"].isin([75, 90, 105, 120])]
dna_run_fil["sample"] = dna_run_fil["sample"].str[1:2]
read_counts["sample"] = read_counts["filename"].str[1:2]
read_counts = read_counts.drop_duplicates(subset="sample")

In [16]:
dna_run_fil["seconds"] = pd.to_timedelta("00:" + dna_run_fil["elapsed"]).dt.total_seconds()
dna_run_fil = dna_run_fil.merge(read_counts, on="sample", how="left")
dna_run_fil["time"] = dna_run_fil["read_count"] / dna_run_fil["seconds"] * 60

In [17]:
snp_run["seconds"] = pd.to_timedelta("00:" + snp_run["elapsed"]).dt.total_seconds()
snp_run["sample"] = "5"
snp_run = snp_run.merge(read_counts, on="sample", how="left")
snp_run["time"] = snp_run["read_count"] / snp_run["seconds"] * 60

In [18]:
dna_run_fil["time"].mean()

13745588.559520679

In [19]:
snp_run["time"].mean()

12250920.002108842